# 🚦 Traffic Sign Detection using YOLOv8
## Final Project - Wuhan Institute of Technology

**Student Name:** [Your Name]  
**Student ID:** [Your ID]  
**Date:** December 2024

---

### 📋 Instructions:
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run each cell from top to bottom
3. Take screenshots at each step for your report
4. Download results at the end

## 📸 STEP 1: Check GPU (Take Screenshot!)

In [ ]:
# ============================================
# STEP 1: GPU & ENVIRONMENT CHECK
# ============================================
# 📸 SCREENSHOT THIS OUTPUT!

import torch

print("=" * 60)
print("🖥️ GPU & ENVIRONMENT INFORMATION")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ GPU not available! Go to Runtime > Change runtime type > GPU")

print("=" * 60)

# Also show nvidia-smi
!nvidia-smi

## 📦 STEP 2: Install YOLOv8

In [ ]:
# ============================================
# STEP 2: INSTALL ULTRALYTICS YOLOv8
# ============================================

!pip install ultralytics -q

from ultralytics import YOLO
import os

print("\n✅ Ultralytics YOLOv8 installed successfully!")
print(f"📦 ultralytics version: {__import__('ultralytics').__version__}")

## 📥 STEP 3: Download GTSDB Dataset

In [ ]:
# ============================================
# STEP 3: DOWNLOAD GTSDB DATASET FROM ROBOFLOW
# ============================================
# Using a pre-formatted GTSDB dataset from Roboflow

!pip install roboflow -q

from roboflow import Roboflow

# Download German Traffic Sign Dataset (GTSDB format)
# This is a public dataset available for educational use
rf = Roboflow(api_key="YOUR_API_KEY")  # Free signup at roboflow.com

# Alternative: Use built-in COCO128 dataset for demo
print("\n📥 Downloading sample dataset for training demo...")
print("\nFor full GTSDB dataset:")
print("1. Go to: https://universe.roboflow.com/")
print("2. Search: 'German Traffic Sign'")
print("3. Download in YOLOv8 format")

In [ ]:
# ============================================
# ALTERNATIVE: CREATE SAMPLE DATASET FOR DEMO
# ============================================
# If you don't have Roboflow API, use COCO128 for demo

# Create data.yaml for GTSDB classes
data_yaml = """
# GTSDB Dataset Configuration
# German Traffic Sign Detection Benchmark

path: /content/datasets/gtsdb
train: images/train
val: images/val

# Classes (4 super-categories)
names:
  0: prohibitory
  1: mandatory
  2: danger
  3: other

nc: 4
"""

# For demo, we'll use COCO128 (comes with YOLOv8)
print("Using COCO128 dataset for training demonstration")
print("(Replace with GTSDB for actual project)")

DATASET = "coco128.yaml"  # Built-in dataset for demo

## 🏋️ STEP 4: BASELINE MODEL TRAINING (Take Screenshot!)

In [ ]:
# ============================================
# STEP 4: TRAIN BASELINE MODEL
# ============================================
# 📸 SCREENSHOT THIS OUTPUT!

print("=" * 60)
print("🏋️ BASELINE MODEL TRAINING")
print("=" * 60)
print("Configuration:")
print("  • Model: YOLOv8n (nano)")
print("  • Image Size: 640x640")
print("  • Epochs: 10 (demo) / 100 (full)")
print("  • Batch Size: 16")
print("=" * 60)

# Load pre-trained model
model = YOLO('yolov8n.pt')

# Train baseline model
# For demo: 10 epochs, For full training: 100 epochs
results_baseline = model.train(
    data=DATASET,           # Dataset config
    epochs=10,              # Use 100 for full training
    imgsz=640,              # Baseline resolution
    batch=16,               # Batch size
    name='baseline',        # Experiment name
    patience=20,            # Early stopping
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ BASELINE TRAINING COMPLETE!")
print("=" * 60)

## 🚀 STEP 5: IMPROVED MODEL TRAINING (Take Screenshot!)

In [ ]:
# ============================================
# STEP 5: TRAIN IMPROVED MODEL (PROPOSED METHOD)
# ============================================
# 📸 SCREENSHOT THIS OUTPUT!

print("=" * 60)
print("🚀 IMPROVED MODEL TRAINING (PROPOSED METHOD)")
print("=" * 60)
print("IMPROVEMENTS:")
print("  ✓ Higher Resolution: 800x800 (+56% pixels)")
print("  ✓ Rotation: ±15° (tilted signs)")
print("  ✓ Mixup: 0.15 (virtual oversampling)")
print("  ✓ Copy-Paste: 0.1 (minority class boost)")
print("  ✓ Cosine LR: Smooth learning rate decay")
print("=" * 60)

# Load fresh model
model_improved = YOLO('yolov8n.pt')

# Train with improvements
results_improved = model_improved.train(
    data=DATASET,
    epochs=10,              # Use 100 for full training
    imgsz=800,              # IMPROVEMENT 1: Higher resolution
    batch=8,                # Reduced for memory
    name='improved',
    patience=25,

    # IMPROVEMENT 2: Enhanced augmentation
    degrees=15.0,           # Rotation ±15°
    shear=5.0,              # Shear ±5°
    perspective=0.001,      # Perspective distortion
    hsv_h=0.02,             # Hue variation
    hsv_s=0.8,              # Saturation
    hsv_v=0.5,              # Brightness

    # IMPROVEMENT 3: Class balancing
    mixup=0.15,             # Mixup augmentation
    copy_paste=0.1,         # Copy-paste augmentation
    mosaic=1.0,             # Mosaic
    close_mosaic=5,         # Disable last 5 epochs

    # Better LR schedule
    cos_lr=True,            # Cosine annealing
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ IMPROVED MODEL TRAINING COMPLETE!")
print("=" * 60)

## 📊 STEP 6: COMPARE RESULTS (Take Screenshot!)

In [ ]:
# ============================================
# STEP 6: COMPARE BASELINE vs IMPROVED
# ============================================
# 📸 SCREENSHOT THIS TABLE!

import pandas as pd

print("=" * 70)
print("📊 PERFORMANCE COMPARISON: BASELINE vs IMPROVED")
print("=" * 70)

# Load results
baseline_model = YOLO('runs/detect/baseline/weights/best.pt')
improved_model = YOLO('runs/detect/improved/weights/best.pt')

# Validate both models
print("\n📈 Validating Baseline Model...")
baseline_results = baseline_model.val(data=DATASET, verbose=False)

print("📈 Validating Improved Model...")
improved_results = improved_model.val(data=DATASET, verbose=False)

# Create comparison table
print("\n" + "=" * 70)
print("\n📋 RESULTS COMPARISON TABLE")
print("\n┌─────────────────┬──────────┬──────────┬─────────────┐")
print("│ Metric          │ Baseline │ Improved │ Improvement │")
print("├─────────────────┼──────────┼──────────┼─────────────┤")

metrics = [
    ('Precision', baseline_results.box.mp, improved_results.box.mp),
    ('Recall', baseline_results.box.mr, improved_results.box.mr),
    ('mAP@0.5', baseline_results.box.map50, improved_results.box.map50),
    ('mAP@0.5:0.95', baseline_results.box.map, improved_results.box.map),
]

for name, base, imp in metrics:
    diff = imp - base
    pct = (diff / base * 100) if base > 0 else 0
    print(f"│ {name:<15} │ {base:.3f}    │ {imp:.3f}    │ +{pct:.1f}%        │")

print("└─────────────────┴──────────┴──────────┴─────────────┘")
print("\n" + "=" * 70)

## 🎯 STEP 7: TEST DETECTION (Take Screenshot!)

In [ ]:
# ============================================
# STEP 7: RUN DETECTION ON TEST IMAGE
# ============================================
# 📸 SCREENSHOT THE DETECTION RESULT!

from google.colab import files
import matplotlib.pyplot as plt
from PIL import Image

print("=" * 60)
print("🎯 REAL-TIME DETECTION TEST")
print("=" * 60)

# Download a sample image for testing
!wget -q -O test_image.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e8/E-road_sign_example.svg/800px-E-road_sign_example.svg.png"

# Or upload your own image
# uploaded = files.upload()

# Run detection with improved model
improved_model = YOLO('runs/detect/improved/weights/best.pt')
results = improved_model('test_image.jpg', save=True, conf=0.25)

# Show results
print("\n📍 Detection Results:")
for result in results:
    boxes = result.boxes
    print(f"   Found {len(boxes)} object(s)")
    for box in boxes:
        conf = float(box.conf[0])
        cls = int(box.cls[0])
        print(f"   • Class {cls}: {conf:.1%} confidence")

# Display annotated image
annotated = results[0].plot()
plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.axis('off')
plt.title('Detection Result - Improved YOLOv8 Model')
plt.show()

print("\n" + "=" * 60)

## 📈 STEP 8: SHOW TRAINING CURVES (Take Screenshot!)

In [ ]:
# ============================================
# STEP 8: DISPLAY TRAINING CURVES
# ============================================
# 📸 SCREENSHOT THESE GRAPHS!

from IPython.display import Image, display
import os

print("=" * 60)
print("📈 TRAINING RESULTS VISUALIZATION")
print("=" * 60)

# Show baseline results
print("\n📊 BASELINE MODEL RESULTS:")
if os.path.exists('runs/detect/baseline/results.png'):
    display(Image(filename='runs/detect/baseline/results.png', width=800))

# Show improved results
print("\n📊 IMPROVED MODEL RESULTS:")
if os.path.exists('runs/detect/improved/results.png'):
    display(Image(filename='runs/detect/improved/results.png', width=800))

## 📥 STEP 9: DOWNLOAD RESULTS

In [ ]:
# ============================================
# STEP 9: DOWNLOAD ALL RESULTS
# ============================================

import shutil
from google.colab import files

print("=" * 60)
print("📥 PREPARING RESULTS FOR DOWNLOAD")
print("=" * 60)

# Create zip of all results
shutil.make_archive('training_results', 'zip', 'runs/detect')

print("\n✅ Results packaged!")
print("\nContents:")
print("  • baseline/ - Baseline model results")
print("  • improved/ - Improved model results")
print("  • weights/best.pt - Trained models")
print("  • results.png - Training curves")
print("  • confusion_matrix.png")

# Download
print("\n📥 Downloading...")
files.download('training_results.zip')

## ✅ SUMMARY

### Screenshots to take for your report:

1. **Step 1**: GPU info (nvidia-smi)
2. **Step 4**: Baseline training progress
3. **Step 5**: Improved training progress
4. **Step 6**: Comparison table
5. **Step 7**: Detection result image
6. **Step 8**: Training curves (results.png)

### Files to download:
- training_results.zip (contains all model files and graphs)

---

**Good luck with your project!** 🎓